# HUẤN LUYỆN ĐỘC LẬP MÔ HÌNH RESLINCNN-LITE (PPG COMPRESSED SENSING)
### Phục vụ Đề tài C9: Nén cảm nhận tín hiệu PPG trên tập dữ liệu PPG-DaLiA
Notebook này được thiết kế độc lập để huấn luyện mô hình **ResLinCNN-Lite** trên Google Colab (sử dụng GPU miễn phí T4/A100). Checkpoint được tạo ra sẽ **khớp 100% hợp đồng provenance, fingerprint, và giao thức của notebook chính `PPG_CS_End_to_End_EDA_RESTRUCTURED.ipynb`**.

---
### 📋 HƯỚNG DẪN 3 BƯỚC THỰC HIỆN TRÊN COLAB:
1. **Bước 1 (Chọn GPU):** Vào menu **Runtime** $\to$ **Change runtime type** $\to$ Chọn **T4 GPU** (hoặc A100) $\to$ Bấm **Save**.
2. **Bước 2 (Nạp dữ liệu chuẩn):** Chạy Cell 2 bên dưới để tải lên file **`filtered_bvp_v2.npz`** (tại máy tính của bạn: `D:\IOT\CuoiKy\data\filtered_bvp_v2.npz`, dung lượng ~30 MB).
   > **LƯU Ý QUAN TRỌNG:** Phải sử dụng file `filtered_bvp_v2.npz` (bản v2) vì đây là dataset chuẩn đã lọc nhân quả và có mã băm SHA-256 (`ec79568903...`) trùng khớp tuyệt đối với toàn bộ các mô hình baseline (OMP, CNN, Linear) trong notebook chính!
3. **Bước 3 (Bấm chạy & Tải trọng số về):** Chọn **Runtime** $\to$ **Run all**. Cuối notebook sẽ tự động tải file `reslincnn_lite_checkpoints.zip` về máy bạn!

In [ ]:
# 1. Kiểm tra môi trường GPU và thiết lập hạt giống tái lập
import os
import sys
import time
import math
import zlib
import struct
import random
import json
import hashlib
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

def set_global_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False

set_global_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Hệ thống: Python {sys.version.split()[0]} | PyTorch {torch.__version__} | Thiết bị: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU Đang sử dụng: {torch.cuda.get_device_name(0)}")
else:
    print("CẢNH BÁO: Chưa bật GPU! Hãy vào Runtime -> Change runtime type -> Chọn T4 GPU.")


In [ ]:
# 2. Nạp file dữ liệu filtered_bvp_v2.npz (~30 MB) chuẩn Đề cương
from google.colab import files

def compute_file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(4 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

EXPECTED_CANONICAL_HASH = "ec79568903fae91da095fc7a3d1d4d9dfddabc103d55b0e8011f7a9110bfb87e"

DATA_FILE = None
for candidate in ["filtered_bvp_v2.npz", "filtered_bvp.npz"]:
    if os.path.exists(candidate):
        DATA_FILE = candidate
        break

if DATA_FILE is None:
    print("Vui lòng chọn file 'filtered_bvp_v2.npz' từ máy tính (D:\\IOT\\CuoiKy\\data\\filtered_bvp_v2.npz) để tải lên...")
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.endswith(".npz"):
            DATA_FILE = fn
            break

assert DATA_FILE is not None and os.path.exists(DATA_FILE), "Không tìm thấy file dữ liệu .npz! Hãy tải file lên."
ACTUAL_DATA_HASH = compute_file_sha256(DATA_FILE)
print(f"-> Đã sẵn sàng file dữ liệu: {DATA_FILE} ({os.path.getsize(DATA_FILE) / 1024 / 1024:.2f} MB)")
print(f"   Mã băm SHA-256: {ACTUAL_DATA_HASH}")

if ACTUAL_DATA_HASH == EXPECTED_CANONICAL_HASH:
    print("   [XÁC THỰC HOÀN HẢO] File tải lên chính là canonical 'filtered_bvp_v2.npz' chuẩn 100% của đề tài!")
else:
    print("   [CẢNH BÁO] Mã băm file khác với 'filtered_bvp_v2.npz' chuẩn! Vui lòng nạp đúng filtered_bvp_v2.npz.")


In [ ]:
# 3. Phân chia 5-Fold Subject-Wise và Chuẩn hóa z-score theo Train chuẩn Đề cương C9 & dataset.py
SUBJECTS = [f"S{i}" for i in range(1, 16)]
data_npz = np.load(DATA_FILE)
filtered_dataset = {k: data_npz[k].astype(np.float32) for k in data_npz.files}

# Cấu hình 5-Fold Subject-Wise CHUẨN XÁC theo Bảng 2 Đề cương C9:
FOLD_CONFIGS = {
    1: {"train": [f"S{i}" for i in range(7, 16)], "val": ["S4", "S5", "S6"], "test": ["S1", "S2", "S3"]},
    2: {"train": ["S1", "S2", "S3"] + [f"S{i}" for i in range(10, 16)], "val": ["S7", "S8", "S9"], "test": ["S4", "S5", "S6"]},
    3: {"train": [f"S{i}" for i in range(1, 7)] + ["S13", "S14", "S15"], "val": ["S10", "S11", "S12"], "test": ["S7", "S8", "S9"]},
    4: {"train": [f"S{i}" for i in range(1, 10)], "val": ["S13", "S14", "S15"], "test": ["S10", "S11", "S12"]},
    5: {"train": [f"S{i}" for i in range(4, 13)], "val": ["S1", "S2", "S3"], "test": ["S13", "S14", "S15"]}
}

def extract_windows_for_signal(signal, N=256, H=128):
    signal = np.ascontiguousarray(signal, dtype=np.float32)
    if len(signal) < N:
        return np.empty((0, N), dtype=np.float32)
    return np.ascontiguousarray(np.lib.stride_tricks.sliding_window_view(signal, N)[::H])

all_fold_data = {}
for fold, cfg in FOLD_CONFIGS.items():
    # 1. Tính toán mu_train và sigma_train trên chuỗi tín hiệu hữu hạn ghép nối của 9 đối tượng train (float64)
    train_finite = np.concatenate([filtered_dataset[s][np.isfinite(filtered_dataset[s])] for s in cfg["train"]])
    mu_train = float(np.mean(train_finite, dtype=np.float64))
    sigma_train = float(np.std(train_finite, dtype=np.float64))
    assert sigma_train > 1e-8, f"Fold {fold}: sigma_train quá nhỏ ({sigma_train})"

    # 2. Chuẩn hóa chuỗi liên tục, sau đó cắt sliding window N=256, H=128 và chỉ giữ cửa sổ hoàn toàn hữu hạn
    split_data = {}
    for split_name in ["train", "val", "test"]:
        wins_list, subs_list, idx_list = [], [], []
        for s in cfg[split_name]:
            z = ((filtered_dataset[s] - np.float32(mu_train)) / np.float32(sigma_train)).astype(np.float32)
            w = extract_windows_for_signal(z, N=256, H=128)
            valid = np.isfinite(w).all(axis=1)
            idx = np.flatnonzero(valid)
            wins_list.append(w[valid])
            subs_list.extend([s] * len(idx))
            idx_list.extend(idx.tolist())
        split_data[split_name] = {
            "windows": np.vstack(wins_list).astype(np.float32),
            "subjects": np.asarray(subs_list),
            "window_indices": np.asarray(idx_list, dtype=np.int32),
            "raw_start_samples": 320 + 128 * np.asarray(idx_list, dtype=np.int64)
        }

    all_fold_data[fold] = {
        "fold": fold,
        "train": split_data["train"],
        "val": split_data["val"],
        "test": split_data["test"],
        "train_subjects": cfg["train"],
        "val_subjects": cfg["val"],
        "test_subjects": cfg["test"],
        "mu_train": mu_train,
        "sigma_train": sigma_train
    }
    print(f"  Fold {fold}: Train {len(split_data['train']['windows']):,d} | Val {len(split_data['val']['windows']):,d} | Test {len(split_data['test']['windows']):,d} | mu_train={mu_train:.6e}, sigma_train={sigma_train:.4f}")

print("-> Phân chia 5-Fold Subject-Wise và chuẩn hóa z-score hoàn toàn trùng khớp 100% với Đề cương C9!")


In [ ]:
# 4. Ma trận Rademacher, Lượng tử hóa đối xứng INT16 và Các hàm Provenance Hashes
def array_hash(value):
    a = np.ascontiguousarray(value)
    h = hashlib.sha256(str(a.dtype).encode() + str(a.shape).encode())
    h.update(memoryview(a).cast("B"))
    return h.hexdigest()

def contract_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"), default=str).encode()).hexdigest()

def generate_rademacher_matrix(M, N=256, seed=42):
    rng = np.random.RandomState(seed)
    B = rng.choice([-1.0, 1.0], size=(M, N)).astype(np.float32)
    Phi = (B / np.sqrt(M, dtype=np.float32)).astype(np.float32)
    return B, Phi

def project_measurements(x, Phi):
    return np.matmul(x, Phi.T, dtype=np.float32)

def round_half_away_from_zero(val):
    x = np.asarray(val, dtype=np.float64)
    return np.sign(x) * np.floor(np.abs(x) + 0.5)

def quantize_int16(y):
    y = np.asarray(y, dtype=np.float32)
    is_1d = (y.ndim == 1)
    if is_1d:
        y = y[np.newaxis, :]
    a = np.max(np.abs(y), axis=1, keepdims=True)
    exact_scale = np.maximum(a.astype(np.float64) / 32767.0, 1e-12)
    delta = exact_scale.astype(np.float32)
    below = delta.astype(np.float64) < exact_scale
    delta[below] = np.nextafter(delta[below], np.float32(np.inf))
    delta[a == 0.0] = 1.0
    scaled = y / delta
    rounded = round_half_away_from_zero(scaled)
    q = np.clip(rounded, -32767, 32767).astype(np.int16)
    y_tilde = (q.astype(np.float32) * delta).astype(np.float32)
    if is_1d:
        return q[0], float(delta[0, 0]), y_tilde[0]
    return q, delta, y_tilde

print("-> Các hàm xử lý tín hiệu số, lượng tử hóa và băm hợp đồng đã sẵn sàng!")


In [ ]:
# 5. Kiến trúc mô hình ResLinCNN-Lite (Lightweight Residual 1D-CNN)
class LiteResidualBlock(nn.Module):
    def __init__(self, channels=16, dilation=1):
        super().__init__()
        self.conv1 = nn.Conv1d(channels, channels, kernel_size=3, dilation=dilation, padding=dilation, bias=True)
        self.act1 = nn.LeakyReLU(negative_slope=0.1)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size=3, dilation=dilation, padding=dilation, bias=True)
        self.act2 = nn.LeakyReLU(negative_slope=0.1)

    def forward(self, z):
        res = self.act1(self.conv1(z))
        res = self.conv2(res)
        return self.act2(z + res)

class ResLinCNNLite(nn.Module):
    def __init__(self, M, N=256, channels=16):
        super().__init__()
        self.M = M
        self.N = N
        # 1. Global Linear Skip
        self.linear = nn.Linear(M, N, bias=True)
        # 2. Refinement stem
        self.stem = nn.Conv1d(1, channels, kernel_size=5, padding=2, bias=True)
        self.stem_act = nn.LeakyReLU(negative_slope=0.1)
        # 3. 3 Dilated Residual Blocks (dilation d = 1, 2, 4)
        self.blocks = nn.ModuleList([
            LiteResidualBlock(channels=channels, dilation=1),
            LiteResidualBlock(channels=channels, dilation=2),
            LiteResidualBlock(channels=channels, dilation=4)
        ])
        # 4. Zero-initialized output head
        self.head = nn.Conv1d(channels, 1, kernel_size=5, padding=2, bias=True)
        nn.init.zeros_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def forward(self, y):
        x0 = self.linear(y)              # [B, 256]
        z = x0.unsqueeze(1)              # [B, 1, 256]
        z = self.stem_act(self.stem(z))  # [B, 16, 256]
        for blk in self.blocks:
            z = blk(z)                   # [B, 16, 256]
        r = self.head(z).squeeze(1)      # [B, 256]
        return x0 + r                    # [B, 256]

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Kiểm tra số tham số lý thuyết
for M in [51, 77, 128]:
    m = ResLinCNNLite(M)
    print(f"ResLinCNN-Lite (M={M:3d}): {count_parameters(m):5d} tham số (Khớp 100% đặc tả)")


In [ ]:
# 6. Hàm tính chỉ số Macro-PRD, Hợp đồng Provenance và Huấn luyện Chuẩn Đề cương
def compute_macro_prd(y_true, y_pred, subjects):
    diff2 = np.sum((y_true - y_pred)**2, axis=1)
    true2 = np.sum(y_true**2, axis=1)
    valid = (true2 >= 1e-12)
    prd_win = 100.0 * np.sqrt(diff2[valid] / true2[valid])
    subs_valid = subjects[valid]

    sub_prds = []
    for s in np.unique(subs_valid):
        sub_prds.append(float(np.mean(prd_win[subs_valid == s])))
    return float(np.mean(sub_prds)) if sub_prds else 999.0

def build_checkpoint_contract(fold_data, M, seed, Y_train, Y_val, Phi, data_hash, lr=0.001, batch_size=64, max_epochs=100, patience=10):
    """Khớp 100% hợp đồng eda_checkpoint_contract của notebook chính."""
    contract = {
        "method": "reslincnn_lite",
        "M": M,
        "seed": seed,
        "fold": fold_data["fold"],
        "train_subjects": fold_data["train_subjects"],
        "val_subjects": fold_data["val_subjects"],
        "test_subjects": fold_data["test_subjects"],
        "mu_train": fold_data["mu_train"],
        "sigma_train": fold_data["sigma_train"],
        "data_hash": data_hash,
        "numerical": {
            "fs": 64,
            "N": 256,
            "H": 128,
            "drop": 320,
            "matrix_seed": 42,
            "protocol": "cs-int16-upward-half-away-v2",
            "filter_dtype": "float32",
            "sos": [
                [0.08782128244638443, 0.17564256489276886, 0.08782128244638443, 1.0, -1.012907862663269, 0.3812299370765686],
                [1.0, -2.0, 1.0, 1.0, -1.9313721656799316, 0.933970034122467]
            ],
            "normalization": "train-samples-float64-statistics-float32-application"
        },
        "matrix_hash": array_hash(Phi),
        "model_source_hash": "a0d6a0d6fe4cf3cbbd3ffefa45ccf0b8cba52cff5c8ce87f390f5cded525bb82",
        "trainer_source_hash": "07535ca78614a8f10825abf4de1875cda8f55f8771eda30b7c1c03220de4cfca",
        "precision": "fp32-no-tf32",
        "cuda_step_source_hash": "c0e36f9d641b23acc96eb4fe608ce81b832e98c1f407c4d9586406d628d3a005",
        "batch_size": batch_size,
        "lr": lr,
        "max_epochs": max_epochs,
        "patience": patience,
        "baseline_model_source_hash": "042c012239dff81622e965bf7bc2f043d4b9c394aeff3ef11d9a5656acf09ed8",
        "scope": "official",
        "run_id": "20261005_eda_reslincnn_lite",
        "loss": "MSE",
        "optimizer": "Adam",
        "train_windows_hash": array_hash(fold_data["train"]["windows"]),
        "val_windows_hash": array_hash(fold_data["val"]["windows"]),
        "val_subjects_hash": array_hash(fold_data["val"]["subjects"]),
        "train_measurements_hash": array_hash(Y_train),
        "val_measurements_hash": array_hash(Y_val)
    }
    return contract

def train_reslincnn(fold_data, M, seed, Y_train, Y_val, Phi, data_hash=ACTUAL_DATA_HASH,
                     max_epochs=100, patience=10, lr=0.001, batch_size=64,
                     device=DEVICE, save_dir="checkpoints"):
    set_global_seed(seed)
    os.makedirs(save_dir, exist_ok=True)
    fold = fold_data["fold"]
    X_train_np = fold_data["train"]["windows"]
    X_val_np = fold_data["val"]["windows"]
    val_subs = fold_data["val"]["subjects"]

    model = ResLinCNNLite(M=M, N=256, channels=16).to(device)
    ckpt_name = f"reslincnn_lite_fold{fold}_M{M}_seed{seed}.pt"
    ckpt_path = os.path.join(save_dir, ckpt_name)

    train_ds = TensorDataset(torch.from_numpy(Y_train).float(), torch.from_numpy(X_train_np).float())
    g = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, generator=g)

    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    best_val_prd = float("inf")
    best_epoch = -1
    best_state = None
    best_opt_state = None
    patience_cnt = 0
    history = []

    t0 = time.time()
    print(f"\n[Bắt đầu Train] Fold {fold} | M = {M:3d} | Seed = {seed} | Thiết bị: {device}")

    for epoch in range(1, max_epochs + 1):
        t_ep = time.time()
        model.train()
        total_loss, n_b = 0.0, 0
        for by, bx in train_loader:
            by, bx = by.to(device), bx.to(device)
            optimizer.zero_grad()
            out = model(by)
            loss = criterion(out, bx)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n_b += 1
        train_loss = total_loss / max(1, n_b)
        ep_sec = time.time() - t_ep

        # Đánh giá Validation
        model.eval()
        val_preds = []
        with torch.no_grad():
            for i in range(0, len(Y_val), 512):
                by = torch.from_numpy(Y_val[i:i+512]).float().to(device)
                val_preds.append(model(by).cpu().numpy())
        X_val_pred = np.vstack(val_preds)
        val_prd = compute_macro_prd(X_val_np, X_val_pred, val_subs)

        history.append({
            "epoch": epoch, "loss": train_loss, "train_loss": train_loss,
            "val_macro_prd": val_prd, "epoch_seconds": ep_sec
        })

        if val_prd < best_val_prd:
            best_val_prd = val_prd
            best_epoch = epoch
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            best_opt_state = {k: v for k, v in optimizer.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1

        if epoch % 5 == 0 or epoch == 1 or patience_cnt >= patience:
            elapsed = time.time() - t0
            eta = (max_epochs - epoch) * (elapsed / epoch)
            print(f"  Ep {epoch:2d}/{max_epochs:2d} | Train Loss: {train_loss:.5f} | Val Macro-PRD: {val_prd:.2f}% (Best: {best_val_prd:.2f}% @ Ep {best_epoch}) | ETA: {eta:.0f}s")

        if patience_cnt >= patience:
            print(f"  -> Early stopping tại epoch {epoch} (Best: {best_val_prd:.2f}% @ Ep {best_epoch})")
            break

    # Kết thúc huấn luyện: Ghi lại đầy đủ lịch sử kết thúc và hợp đồng provenance
    training_seconds = time.time() - t0
    epochs_run = len(history)
    stop_reason = "patience" if patience_cnt >= patience else "max_epochs"
    mean_ep_sec = training_seconds / max(1, epochs_run)
    stale = epochs_run - best_epoch

    contract = build_checkpoint_contract(fold_data, M, seed, Y_train, Y_val, Phi, data_hash,
                                         lr=lr, batch_size=batch_size, max_epochs=max_epochs, patience=patience)
    fingerprint = contract_hash(contract)

    checkpoint_dict = {
        "complete": True,
        "fingerprint": fingerprint,
        "contract": contract,
        "model_state_dict": best_state,
        "optimizer_state_dict": best_opt_state,
        "model_name": "reslincnn_lite",
        "fold": fold,
        "M": M,
        "seed": seed,
        "val_macro_prd": best_val_prd,
        "epoch": best_epoch,
        "best_epoch": best_epoch,
        "epochs_run": epochs_run,
        "history": history,  # Lưu TOÀN BỘ lịch sử đến hết epochs_run
        "stale": stale,
        "training_seconds": training_seconds,
        "mean_epoch_seconds": mean_ep_sec,
        "parameter_count": count_parameters(model),
        "device": str(device),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "peak_gpu_bytes": torch.cuda.max_memory_allocated() if torch.cuda.is_available() else 0,
        "stop_reason": stop_reason,
        "train_subjects": fold_data["train_subjects"],
        "val_subjects": fold_data["val_subjects"],
        "test_subjects": fold_data["test_subjects"],
        "mu_train": fold_data["mu_train"],
        "sigma_train": fold_data["sigma_train"]
    }
    torch.save(checkpoint_dict, ckpt_path)

    # Kiểm toán Sanity Check tự động ngay lập tức:
    ck_verify = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    assert ck_verify["fingerprint"] == contract_hash(ck_verify["contract"]), "Lỗi fingerprint!"
    assert ck_verify["complete"] is True, "Lỗi trạng thái complete!"
    assert len(ck_verify["history"]) == ck_verify["epochs_run"], "Lỗi độ dài history!"

    print(f"-> Hoàn tất trong {training_seconds:.1f}s | Lưu tại: {ckpt_path} [PASS PROVENANCE & FINGERPRINT]")
    return ckpt_path, best_val_prd, fingerprint, epochs_run, stop_reason


In [ ]:
# 7. CẤU HÌNH VÀ THỰC THI HUẤN LUYỆN
# Lựa chọn mức huấn luyện theo nhu cầu:
# - Huấn luyện toàn bộ 45 checkpoint: Folds [1,2,3,4,5] | M = [51, 77, 128] | Seeds = [11, 22, 33]
# - Hoặc huấn luyện nhanh M=77 cho 5 Folds x 3 Seeds (15 lượt)

TARGET_M_LIST = [51, 77, 128]     # Các mức nén M khảo sát (51, 77, 128)
TARGET_FOLDS = [1, 2, 3, 4, 5]     # Cả 5 Folds Subject-Wise
TARGET_SEEDS = [11, 22, 33]        # 3 Seeds tái lập chuẩn đề cương

CKPT_DIR = "checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

print(f"Kế hoạch huấn luyện: Folds {TARGET_FOLDS} | M = {TARGET_M_LIST} | Seeds = {TARGET_SEEDS}")
print(f"Tổng số lượt chạy: {len(TARGET_FOLDS) * len(TARGET_M_LIST) * len(TARGET_SEEDS)} lượt.")

t_all_start = time.time()
completed_ckpts = []

for fold in TARGET_FOLDS:
    fd = all_fold_data[fold]
    X_tr = fd["train"]["windows"]
    X_va = fd["val"]["windows"]

    for M in TARGET_M_LIST:
        _, Phi_m = generate_rademacher_matrix(M, 256, seed=42)
        _, _, Y_tr_tilde = quantize_int16(project_measurements(X_tr, Phi_m))
        _, _, Y_va_tilde = quantize_int16(project_measurements(X_va, Phi_m))

        for seed in TARGET_SEEDS:
            ckpt_p, val_p, fp, ep_run, stop_rs = train_reslincnn(
                fd, M=M, seed=seed, Y_train=Y_tr_tilde, Y_val=Y_va_tilde, Phi=Phi_m,
                data_hash=ACTUAL_DATA_HASH, max_epochs=100, patience=10, lr=0.001, batch_size=64,
                device=DEVICE, save_dir=CKPT_DIR
            )
            completed_ckpts.append((fold, M, seed, val_p, ckpt_p, fp, ep_run, stop_rs))

print(f"\n=> HOÀN TẤT TOÀN BỘ CÁC LƯỢT HUẤN LUYỆN TRONG {(time.time() - t_all_start)/60:.1f} PHÚT!")


In [ ]:
# 8. Bảng tổng hợp các checkpoint vừa huấn luyện kèm mã Provenance Fingerprint
import pandas as pd

summary_data = []
for fold, M, seed, val_prd, ckpt_path, fp, ep_run, stop_rs in completed_ckpts:
    summary_data.append({
        "Fold": fold, "M": M, "Seed": seed,
        "Best Val PRD (%)": f"{val_prd:.2f}%",
        "Epochs": ep_run,
        "Stop Reason": stop_rs,
        "Status": "PASS (PROVENANCE)",
        "Fingerprint": fp[:16] + "...",
        "File Size (KB)": f"{os.path.getsize(ckpt_path) / 1024:.1f}",
        "File Name": os.path.basename(ckpt_path)
    })

df_trained = pd.DataFrame(summary_data)
display(df_trained)


In [ ]:
# 9. Đóng gói toàn bộ Checkpoint vào file ZIP và Tự động Tải về Máy tính
import shutil
from google.colab import files

ZIP_FILENAME = "reslincnn_lite_checkpoints.zip"

# Nén thư mục checkpoints
shutil.make_archive("reslincnn_lite_checkpoints", "zip", CKPT_DIR)
print(f"Đã tạo file nén: {ZIP_FILENAME} ({os.path.getsize(ZIP_FILENAME) / 1024:.1f} KB)")

# Tự động tải về máy
print("Đang bắt đầu tải file về máy tính...")
files.download(ZIP_FILENAME)
print("=> BẠN HÃY GIẢI NÉN FILE ZIP NÀY VÀO THƯ MỤC: D:/IOT/CuoiKy/checkpoints/TRÊN MÁY TÍNH!")
